# Hospital Readmission Prediction — Exploration & Modeling
Case Study: Logistic Regression with L2 Regularization

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, RocCurveDisplay, ConfusionMatrixDisplay

df = pd.read_csv('../data/raw.csv')
df.head()

## 1. Exploratory Data Analysis

In [ ]:
df.info()
df.describe()

In [ ]:
print('Readmission rate:', df['readmitted_30_days'].mean())
df['readmitted_30_days'].value_counts().plot(kind='bar', title='Readmission Class Balance')
plt.show()

In [ ]:
df['diagnosis_code'].value_counts().plot(kind='bar', title='Diagnosis Code Distribution')
plt.show()

## 2. Preprocessing

In [ ]:
df_encoded = pd.get_dummies(df, columns=['gender', 'diagnosis_code'], drop_first=True)
df_encoded.to_csv('../data/processed.csv', index=False)

X = df_encoded.drop(columns=['readmitted_30_days'])
y = df_encoded['readmitted_30_days']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Train Logistic Regression with L2 Regularization + Cross-Validation

In [ ]:
param_grid = {'C': [0.001, 0.01, 0.1, 1, 10, 100]}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

l2_model = LogisticRegression(penalty='l2', solver='lbfgs', max_iter=1000)
grid_l2 = GridSearchCV(l2_model, param_grid, cv=cv, scoring='roc_auc')
grid_l2.fit(X_train_scaled, y_train)

print('Best C (L2):', grid_l2.best_params_['C'])
print('Best CV ROC-AUC:', grid_l2.best_score_)

best_l2_model = grid_l2.best_estimator_

## 4. Compare with L1 Regularization

In [ ]:
l1_model = LogisticRegression(penalty='l1', solver='liblinear', max_iter=1000)
grid_l1 = GridSearchCV(l1_model, param_grid, cv=cv, scoring='roc_auc')
grid_l1.fit(X_train_scaled, y_train)

print('Best C (L1):', grid_l1.best_params_['C'])
print('Best CV ROC-AUC:', grid_l1.best_score_)

best_l1_model = grid_l1.best_estimator_

## 5. Evaluate on Test Set: Confusion Matrix, Precision/Recall, ROC-AUC

In [ ]:
y_pred = best_l2_model.predict(X_test_scaled)
y_prob = best_l2_model.predict_proba(X_test_scaled)[:, 1]

cm = confusion_matrix(y_test, y_pred)
print(classification_report(y_test, y_pred, target_names=['No Readmit', 'Readmit']))
print('ROC-AUC:', roc_auc_score(y_test, y_prob))

ConfusionMatrixDisplay(cm, display_labels=['No Readmit', 'Readmit']).plot(cmap='Blues')
plt.title('Confusion Matrix - L2 Logistic Regression')
plt.show()

In [ ]:
RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title('ROC Curve - L2 Logistic Regression')
plt.show()

## 6. Clinical Cost Discussion: False Positives vs. False Negatives

A **False Negative** (predicting 'no readmission' when a patient actually IS readmitted) is far more costly than a **False Positive** in this context.

- **False Negative:** A high-risk patient is discharged without extra follow-up, monitoring, or intervention — increasing the risk of complications, a preventable readmission, or patient harm.
- **False Positive:** A lower-risk patient receives some unnecessary follow-up (a phone call, an extra check-in) — costly in staff time, but not dangerous.

**Conclusion:** Hospitals should favor a model tuned for **higher Recall (sensitivity)** even at some cost to Precision — achievable by lowering the classification threshold below 0.5 — since missing a genuinely high-risk patient is clinically worse than a false alarm.

In [ ]:
tn, fp, fn, tp = cm.ravel()
print(f'False Positives: {fp}')
print(f'False Negatives: {fn}')
print(f'\nL2 ROC-AUC: {roc_auc_score(y_test, y_prob):.4f}')